In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                                                                              ║
# ║          🔵  MACHINE LEARNING PARA DATA ANALYSTS  ·  2026                   ║
# ║                                                                              ║
# ║          Ejercicio — CLUSTERING JERÁRQUICO · Dataset Gimnasio               ║
# ║          Versión 2 — 4 variables · Dataset real                             ║
# ║                                                                              ║
# ║          Autor   : Borja Mora Méndez                                         ║
# ║          Dataset : dataset_didactico_machine_learning_ALUMNOS.xlsx           ║
# ║          Features: Antigüedad · Asistencias · Horas Pico · Gasto Extra      ║
# ║          Técnica : Hierarchical Clustering · Ward Linkage                    ║
# ║                                                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# DIFERENCIAS RESPECTO A LA VERSIÓN 1 (2 variables):
# ────────────────────────────────────────────────────
# ✅  4 variables numéricas en lugar de 2
# ✅  Dataset real del curso (no datos sintéticos)
# ✅  Sin visualización scatter directa → heatmap de perfiles + pairplot
# ✅  Comparativa Silhouette: 2 vars vs 4 vars
# ✅  Perfil de negocio completo con las 4 dimensiones
# ✅  Validación cruzada con columna Abandono (no usada en clustering)


---
## 📦 1 · Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings("ignore")

# ── PALETA DE MARCA ────────────────────────────────────────────────────────────
PURPLE      = "#7a7bff"
INK         = "#111111"
MUTED       = "#888888"
GRAY        = "#e5e5e5"
GREEN       = "#3c8a37"
RED         = "#a3223e"
PURPLE_LITE = "#ededff"
CLUSTER_COLORS = ["#7a7bff", "#111111", "#a3223e", "#3c8a37"]

plt.rcParams.update({
    "figure.dpi"        : 130,
    "figure.facecolor"  : "#ffffff",
    "axes.facecolor"    : "#ffffff",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.color"        : "#f0f0f0",
    "grid.linewidth"    : 0.5,
    "font.size"         : 11,
    "xtick.color"       : MUTED,
    "ytick.color"       : MUTED,
})

print("✅ Setup completado")


---
## 📂 2 · Carga y exploración del dataset real

Dataset del curso: **300 clientes de un gimnasio**.

| Variable | Tipo | Descripción |
|---|---|---|
| `Antiguedad_Meses` | int | Meses como socio |
| `Asistencias_Mes` | int | Visitas al gimnasio por mes |
| `Horas_Pico_Mes` | float | Horas de uso en horario pico |
| `Gasto_Mensual_Extra` | float | Gasto adicional en servicios (€) |
| `Satisfecho` | binaria | NO entra en clustering |
| `Abandono` | binaria | NO entra en clustering — es el target |

**Regla:** el clustering solo usa variables de comportamiento observable,  
no identificadores ni variables resultado.


In [ ]:
# ── CARGA ─────────────────────────────────────────────────────────────────────
# Ajusta la ruta a tu entorno local
RUTA = r"dataset_didactico_machine_learning_-_ALUMNOS.xlsx"

data = pd.read_excel(RUTA)

print(f"Shape: {data.shape}")
print(f"\nColumnas: {data.columns.tolist()}")
print(f"\nNulos: {data.isnull().sum().sum()} → dataset limpio")
print()
display(data.head(6))


In [ ]:
# ── ESTADÍSTICAS DESCRIPTIVAS ─────────────────────────────────────────────────
print("📊 ESTADÍSTICAS DESCRIPTIVAS:")
display(data.describe().round(2))

# ── NOTA SOBRE LAS MAGNITUDES ─────────────────────────────────────────────────
print("\n⚠️  PROBLEMA DE MAGNITUDES — por eso normalizar es obligatorio:")
print(f"  Rango Antigüedad_Meses    : {data['Antiguedad_Meses'].min():.0f} – {data['Antiguedad_Meses'].max():.0f}")
print(f"  Rango Asistencias_Mes     : {data['Asistencias_Mes'].min():.0f} – {data['Asistencias_Mes'].max():.0f}")
print(f"  Rango Horas_Pico_Mes      : {data['Horas_Pico_Mes'].min():.1f} – {data['Horas_Pico_Mes'].max():.1f}")
print(f"  Rango Gasto_Mensual_Extra : {data['Gasto_Mensual_Extra'].min():.1f} – {data['Gasto_Mensual_Extra'].max():.1f}")
print("\n→ Sin normalizar, Gasto_Mensual_Extra dominaría todas las distancias")


---
## 📊 3 · Exploración visual — distribuciones

Antes de clusterizar: ¿cómo se distribuye cada variable?  
¿Hay grupos naturales visibles? ¿Hay outliers extremos?


In [ ]:
# ── DISTRIBUCIONES DE LAS 4 FEATURES ─────────────────────────────────────────
FEATURES = ["Antiguedad_Meses", "Asistencias_Mes", "Horas_Pico_Mes", "Gasto_Mensual_Extra"]
LABELS   = ["Antigüedad (meses)", "Asistencias / mes", "Horas pico / mes", "Gasto mensual extra (€)"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax, feat, lbl in zip(axes, FEATURES, LABELS):
    ax.hist(data[feat], bins=25, color=PURPLE, alpha=0.8, edgecolor="none")
    ax.axvline(data[feat].mean(),   color=INK,   linewidth=1.5, linestyle="--", label=f"Media  {data[feat].mean():.1f}")
    ax.axvline(data[feat].median(), color=MUTED, linewidth=1.5, linestyle=":",  label=f"Mediana {data[feat].median():.1f}")
    ax.set_title(lbl, fontsize=12, fontweight="bold", color=INK)
    ax.set_xlabel("")
    ax.legend(fontsize=9, framealpha=0)

plt.suptitle("Distribución de las 4 variables — Dataset Gimnasio (n=300)",
             fontsize=13, fontweight="bold", color=INK)
plt.tight_layout()
plt.show()


In [ ]:
# ── MATRIZ DE CORRELACIÓN ─────────────────────────────────────────────────────
# Importante antes del clustering: ¿hay variables muy correlacionadas?
# Si dos variables dicen lo mismo, una puede ser redundante.

corr = data[FEATURES].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(
    corr,
    annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor=GRAY,
    ax=ax, cbar_kws={"shrink": 0.8}
)
ax.set_title("Matriz de correlación — Features de clustering",
             fontsize=12, fontweight="bold", color=INK, pad=12)
plt.tight_layout()
plt.show()

print("📌 Lectura clave:")
print(f"   Asistencias vs Horas_Pico : {corr.loc['Asistencias_Mes','Horas_Pico_Mes']:.2f}")
print("   → Correlación alta: ambas miden actividad física. El algoritmo las verá como refuerzo mutuo.")
print("   → No es un problema para clustering (sí lo sería en regresión por multicolinealidad).")


---
## 🔧 4 · Selección de variables y normalización

### ¿Por qué estas 4 y no las 7?

| Variable | ¿Entra? | Motivo |
|---|---|---|
| `ID_Cliente` | ❌ | Identificador sin información |
| `Antiguedad_Meses` | ✅ | Comportamiento observable |
| `Asistencias_Mes` | ✅ | Comportamiento observable |
| `Horas_Pico_Mes` | ✅ | Comportamiento observable |
| `Gasto_Mensual_Extra` | ✅ | Comportamiento observable |
| `Satisfecho` | ❌ | Resultado subjetivo, no comportamiento |
| `Abandono` | ❌ | Target — es lo que queremos predecir/entender |

> Si metieras `Abandono` en el clustering, estarías diciéndole al algoritmo  
> la respuesta antes de que busque los patrones. Eso no es aprendizaje no supervisado.


In [ ]:
# ── SELECCIÓN DE FEATURES ─────────────────────────────────────────────────────
FEATURES = ["Antiguedad_Meses", "Asistencias_Mes", "Horas_Pico_Mes", "Gasto_Mensual_Extra"]

X_raw = data[FEATURES].copy()

# ── NORMALIZACIÓN CON StandardScaler ─────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

X_norm = pd.DataFrame(X_scaled, columns=FEATURES, index=data.index)

# ── VERIFICACIÓN ──────────────────────────────────────────────────────────────
print("📊 VERIFICACIÓN DE NORMALIZACIÓN (media ≈ 0 · std ≈ 1):")
print("─" * 50)
print(f"{'Variable':<25} {'Media':>8} {'Std':>8}")
print("─" * 50)
for col in FEATURES:
    print(f"{col:<25} {X_norm[col].mean():>8.4f} {X_norm[col].std():>8.4f}")
print("─" * 50)
print("✅ Todas las variables en la misma escala Z-score")


In [ ]:
# ── COMPARATIVA VISUAL: ANTES vs DESPUÉS ─────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
LABELS_SHORT = ["Antigüedad", "Asistencias", "Horas Pico", "Gasto Extra"]

for col_idx, (feat, lbl) in enumerate(zip(FEATURES, LABELS_SHORT)):

    # Fila superior: originales
    axes[0][col_idx].hist(X_raw[feat], bins=20, color=GRAY, alpha=0.9, edgecolor="none")
    axes[0][col_idx].set_title(f"{lbl}\nOriginal", fontsize=10, fontweight="bold", color=INK)
    axes[0][col_idx].text(0.95, 0.92,
        f"μ={X_raw[feat].mean():.1f}\nσ={X_raw[feat].std():.1f}",
        transform=axes[0][col_idx].transAxes, fontsize=8,
        ha="right", va="top", color=MUTED)

    # Fila inferior: normalizados
    axes[1][col_idx].hist(X_norm[feat], bins=20, color=PURPLE, alpha=0.85, edgecolor="none")
    axes[1][col_idx].set_title(f"{lbl}\nZ-score", fontsize=10, fontweight="bold", color=INK)
    axes[1][col_idx].text(0.95, 0.92,
        f"μ={X_norm[feat].mean():.2f}\nσ={X_norm[feat].std():.2f}",
        transform=axes[1][col_idx].transAxes, fontsize=8,
        ha="right", va="top", color=MUTED)

plt.suptitle("Efecto del StandardScaler — Original (gris) vs Normalizado (morado)",
             fontsize=13, fontweight="bold", color=INK)
plt.tight_layout()
plt.show()


---
## 🌳 5 · Linkage y Dendrograma

Calculamos la matriz de fusiones con **Ward** sobre las 4 variables normalizadas.  
Ward minimiza la varianza interna de cada cluster al fusionar — es el estándar para la industria.

**¿Qué cambia al usar 4 variables en lugar de 2?**  
El dendrograma sigue siendo el mismo tipo de gráfico.  
Lo que cambia es que las distancias ahora se calculan en un espacio de 4 dimensiones  
en lugar de 2 — más información, potencialmente mejores clusters.


In [ ]:
# ── CÁLCULO DEL LINKAGE ───────────────────────────────────────────────────────
Z = linkage(
    X_norm,
    method="ward",       # Minimiza varianza interna — estándar industria
    metric="euclidean"   # Ward solo funciona con distancia euclidea
)

print(f"✅ Linkage calculado sobre {len(FEATURES)} variables normalizadas")
print(f"   Shape de Z: {Z.shape}  ← ({len(data)-1} fusiones × 4 columnas)")
print(f"\n   Últimas 5 fusiones (las más grandes — aquí está el codo):")
print(f"   {'Cluster A':>10} {'Cluster B':>10} {'Distancia':>12} {'Tamaño':>8}")
print("   " + "─"*44)
for row in Z[-5:]:
    print(f"   {int(row[0]):>10} {int(row[1]):>10} {row[2]:>12.4f} {int(row[3]):>8}")


In [ ]:
# ── DENDROGRAMA ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

dendrogram(
    Z,
    ax=ax,
    leaf_rotation=90,
    leaf_font_size=6,
    color_threshold=Z[-3, 2],
    above_threshold_color=MUTED
)

ax.set_title("Dendrograma — Clustering Jerárquico Ward · 4 variables · Clientes Gimnasio",
             fontsize=13, fontweight="bold", color=INK, pad=14)

# Ejes correctos: observaciones / distancia (NO las variables del dataset)
ax.set_xlabel("Clientes (índice de observación)", color=MUTED, fontsize=10)
ax.set_ylabel("Distancia de fusión (Ward)", color=MUTED, fontsize=10)

# Línea de corte para k=3
corte_y = Z[-3, 2] + 0.5
ax.axhline(y=corte_y, color=RED, linewidth=1.8, linestyle="--", alpha=0.85)
ax.text(
    len(data) * 0.70, corte_y + 0.8,
    "← Línea de corte → k = 3 clusters",
    color=RED, fontsize=9, fontweight="bold"
)

ax.text(
    0.01, 0.97,
    "📖 Cómo leer:
"
    "• Cada hoja = 1 cliente (300 total)
"
    "• Altura de la unión = distancia al fusionarse
"
    "• Salto más largo entre ramas = k óptimo
"
    "• Las ramas de color = los k clusters elegidos",
    transform=ax.transAxes, fontsize=8.5, va="top", color=INK,
    bbox=dict(boxstyle="round,pad=0.5", facecolor=PURPLE_LITE, edgecolor="none")
)

plt.tight_layout()
plt.show()


---
## 📐 6 · Método del Codo — ¿Cuántos clusters?

El dendrograma nos da una pista visual. El codo lo confirma numéricamente.  
Buscamos el punto donde la distancia de fusión da el mayor salto:  
ese salto significa que estamos uniendo grupos que **no deberían estar juntos**.


In [ ]:
# ── MÉTODO DEL CODO ───────────────────────────────────────────────────────────
distancias       = Z[:, 2]
n_mostrar        = 10
ultimas          = distancias[-n_mostrar:][::-1]   # Últimas 10, de mayor a menor k
k_vals           = range(1, n_mostrar + 1)

# Aceleración: diferencia entre distancias consecutivas
aceleracion      = np.diff(ultimas)
k_optimo         = int(np.argmax(aceleracion)) + 2  # +2 porque diff reduce en 1 y empezamos en k=1

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Panel izquierdo: curva del codo ───────────────────────────────────────────
ax1 = axes[0]
ax1.plot(
    k_vals, ultimas,
    color=PURPLE, linewidth=2.5, marker="o",
    markersize=8, markerfacecolor="white",
    markeredgecolor=PURPLE, markeredgewidth=2
)
ax1.axvline(x=k_optimo, color=RED, linewidth=1.5, linestyle="--", alpha=0.8)
ax1.scatter([k_optimo], [ultimas[k_optimo - 1]], color=RED, s=130, zorder=5)
ax1.text(
    k_optimo + 0.15, ultimas[k_optimo - 1],
    f"  Codo → k={k_optimo}", color=RED, fontsize=10,
    fontweight="bold", va="center"
)
ax1.set_title("Método del Codo
(distancias de fusión Ward)", fontsize=11, fontweight="bold", color=INK)
ax1.set_xlabel("Número de clusters (k)", color=MUTED, fontsize=10)
ax1.set_ylabel("Distancia de fusión", color=MUTED, fontsize=10)
ax1.set_xticks(k_vals)

# ── Panel derecho: aceleración (salto entre distancias) ───────────────────────
ax2 = axes[1]
bar_colors = [RED if a == aceleracion.max() else PURPLE for a in aceleracion]
ax2.bar(list(k_vals)[1:], aceleracion, color=bar_colors, alpha=0.85, width=0.6)
ax2.set_title("Aceleración entre fusiones
(barra roja = mayor salto = k óptimo)",
              fontsize=11, fontweight="bold", color=INK)
ax2.set_xlabel("Número de clusters (k)", color=MUTED, fontsize=10)
ax2.set_ylabel("Incremento de distancia", color=MUTED, fontsize=10)
ax2.set_xticks(list(k_vals)[1:])

plt.suptitle(f"Selección de k óptimo — k = {k_optimo} clusters",
             fontsize=13, fontweight="bold", color=INK)
plt.tight_layout()
plt.show()

# ── Tabla de distancias ───────────────────────────────────────────────────────
print(f"
📊 TABLA DE DISTANCIAS DE FUSIÓN:")
print(f"{'k':>5} {'Distancia':>12} {'Salto':>12}")
print("─" * 32)
prev = None
for k, d in zip(k_vals, ultimas):
    salto = f"{d - prev:.4f}{'  ← MAYOR ⭐' if prev and (d - prev) == aceleracion.max() else ''}" if prev else "—"
    marca = "  ← ÓPTIMO" if k == k_optimo else ""
    print(f"{k:>5} {d:>12.4f} {salto:>12}{marca}")
    prev = d


---
## 🔬 7 · Comparativa Silhouette: 2 variables vs 4 variables

Antes de asignar los clusters definitivos, respondemos la pregunta clave:  
**¿Añadir las 4 variables mejora realmente la calidad del clustering?**

El **Silhouette Score** mide qué tan bien separados están los clusters.  
Rango: de -1 (pésimo) a +1 (perfecto). Por encima de 0.5 es considerado bueno.


In [ ]:
# ── COMPARATIVA: 2 vars vs 4 vars ─────────────────────────────────────────────
K = k_optimo

experimentos = {
    "2 variables
(V1 del ejercicio)":
        ["Antiguedad_Meses", "Gasto_Mensual_Extra"],
    "4 variables
(V2 — versión completa)":
        ["Antiguedad_Meses", "Asistencias_Mes", "Horas_Pico_Mes", "Gasto_Mensual_Extra"],
}

resultados = {}
print(f"📊 COMPARATIVA SILHOUETTE SCORE (k={K}):")
print("─" * 52)
print(f"{'Versión':<35} {'Silhouette':>12}")
print("─" * 52)

for nombre, feats in experimentos.items():
    Xs = StandardScaler().fit_transform(data[feats])
    modelo = AgglomerativeClustering(n_clusters=K, linkage="ward")
    etiq   = modelo.fit_predict(Xs)
    sil    = silhouette_score(Xs, etiq)
    resultados[nombre] = {"sil": sil, "etiq": etiq, "feats": feats}
    label_clean = nombre.replace("\n", " ")
    print(f"{label_clean:<35} {sil:>12.4f}")

print("─" * 52)
mejor = max(resultados, key=lambda x: resultados[x]["sil"])
print(f"
✅ Mejor versión: {mejor.replace(chr(10), ' ')}")
print(f"   Silhouette: {resultados[mejor]['sil']:.4f}")


---
## 🏷️ 8 · Asignación de clusters con `fcluster`

Con el k elegido, cortamos el dendrograma y asignamos cada cliente a su cluster.


In [ ]:
# ── ASIGNACIÓN FINAL ──────────────────────────────────────────────────────────
K_FINAL = k_optimo

etiquetas = fcluster(Z, t=K_FINAL, criterion="maxclust")
data["Cluster"] = etiquetas

print(f"✅ Clusters asignados con k={K_FINAL}")
print(f"
{'Cluster':>8} {'N clientes':>12} {'%':>8}")
print("─" * 32)
for c, n in data["Cluster"].value_counts().sort_index().items():
    print(f"{c:>8} {n:>12} {n/len(data)*100:>7.1f}%")
print("─" * 32)
print(f"{'Total':>8} {len(data):>12} {'100.0%':>8}")


---
## 💼 9 · Perfil de cada cluster — Las 4 dimensiones

Con 4 variables ya no tenemos un scatter que lo muestre todo.  
Usamos dos herramientas complementarias:

- **Heatmap de perfiles** → vista ejecutiva, de un vistazo
- **Pairplot** → vista analítica, todas las combinaciones de 2 variables


In [ ]:
# ── PERFIL ESTADÍSTICO ────────────────────────────────────────────────────────
perfil = data.groupby("Cluster")[FEATURES + ["Abandono"]].mean().round(2)
perfil["N_clientes"] = data.groupby("Cluster").size()

LABELS_PERFIL = {
    "Antiguedad_Meses"   : "Antigüedad (m)",
    "Asistencias_Mes"    : "Asistencias/mes",
    "Horas_Pico_Mes"     : "Horas pico/mes",
    "Gasto_Mensual_Extra": "Gasto extra (€)",
    "Abandono"           : "% Abandono",
    "N_clientes"         : "N clientes",
}
perfil_display = perfil.rename(columns=LABELS_PERFIL)

print("📊 PERFIL DE CADA CLUSTER (valores medios en escala original):")
print("=" * 72)
display(perfil_display)


In [ ]:
# ── HEATMAP DE PERFILES ───────────────────────────────────────────────────────
# Normalizamos solo para la visualización (escala 0-1)
# Los números anotados son los valores reales originales

perfil_viz   = perfil[FEATURES]
perfil_norm  = (perfil_viz - perfil_viz.min()) / (perfil_viz.max() - perfil_viz.min())

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    perfil_norm.T,
    annot=perfil_viz.T,
    fmt=".1f",
    cmap="RdPu",
    linewidths=0.8,
    linecolor="#f0f0f0",
    cbar_kws={"label": "Intensidad relativa (0=mín · 1=máx)", "shrink": 0.8},
    ax=ax
)

ax.set_title("Perfil de clusters — valores reales anotados · intensidad relativa en color",
             fontsize=12, fontweight="bold", color=INK, pad=12)
ax.set_xlabel("Cluster", color=MUTED, fontsize=10)
ax.set_ylabel("")
ax.set_yticklabels(
    ["Antigüedad (m)", "Asistencias/mes", "Horas pico/mes", "Gasto extra (€)"],
    rotation=0, fontsize=10
)
plt.tight_layout()
plt.show()

print("→ Columna más oscura = cluster más activo en esa variable")
print("→ Columna más clara  = cluster menos activo en esa variable")


In [ ]:
# ── PAIRPLOT — TODAS LAS COMBINACIONES DE 2 VARIABLES ────────────────────────
# Permite ver la separación de clusters en cada par de dimensiones

data_plot = data[FEATURES + ["Cluster"]].copy()
data_plot["Cluster"] = data_plot["Cluster"].astype(str)

palette = {str(i+1): c for i, c in enumerate(CLUSTER_COLORS[:K_FINAL])}

g = sns.pairplot(
    data_plot,
    hue="Cluster",
    palette=palette,
    plot_kws={"alpha": 0.55, "s": 35, "edgecolor": "none"},
    diag_kind="kde",
    corner=True
)

g.fig.suptitle(
    "Pairplot — Separación de clusters en todas las combinaciones de variables",
    fontsize=12, fontweight="bold", color=INK, y=1.01
)

# Renombrar ejes
new_labels = ["Antigüedad (m)", "Asistencias/mes", "Horas pico/mes", "Gasto extra (€)"]
for i, ax in enumerate(g.axes[-1]):
    if ax is not None:
        ax.set_xlabel(new_labels[i], fontsize=9, color=MUTED)
for i, row in enumerate(g.axes):
    if row[0] is not None:
        row[0].set_ylabel(new_labels[i], fontsize=9, color=MUTED)

plt.tight_layout()
plt.show()

print("→ Cada panel = scatter de 2 variables coloreado por cluster")
print("→ Si los colores se separan bien → el cluster tiene sentido en esas 2 dimensiones")


---
## 🎯 10 · Etiquetas de negocio y validación con Abandono

Ahora lo más importante: **darle nombre a cada cluster** basándonos en su perfil.

Y después: ¿coinciden los clusters con quienes abandonan de verdad?  
Eso valida que el algoritmo ha encontrado estructura real en los datos.


In [ ]:
# ── ETIQUETADO AUTOMÁTICO BASADO EN PERFIL ────────────────────────────────────
# Ordenamos los clusters por nivel de actividad general

perfil_activ = perfil[["Asistencias_Mes", "Horas_Pico_Mes", "Gasto_Mensual_Extra"]].mean(axis=1)
orden        = perfil_activ.sort_values().index.tolist()

etiquetas_negocio = {}
nombres           = ["💤 Perfil Inactivo", "⚙️  Perfil Medio", "🏆 Perfil VIP"]

for cluster_id, nombre in zip(orden, nombres):
    etiquetas_negocio[cluster_id] = nombre

data["Segmento"] = data["Cluster"].map(etiquetas_negocio)

# ── PERFIL FINAL CON ETIQUETAS ────────────────────────────────────────────────
perfil_final = data.groupby("Segmento").agg(
    N_Clientes        = ("Cluster",            "count"),
    Antiguedad_Media  = ("Antiguedad_Meses",   "mean"),
    Asistencias_Media = ("Asistencias_Mes",    "mean"),
    Horas_Pico_Media  = ("Horas_Pico_Mes",     "mean"),
    Gasto_Medio       = ("Gasto_Mensual_Extra", "mean"),
    Tasa_Abandono_pct = ("Abandono",           lambda x: x.mean() * 100)
).round(1)

print("📊 PERFIL FINAL CON ETIQUETAS DE NEGOCIO:")
print("=" * 75)
display(perfil_final)

print("
💡 VALIDACIÓN CRÍTICA:")
print("   La columna 'Tasa_Abandono_pct' NO se usó para crear los clusters.")
print("   Si los clusters con menos actividad tienen más abandono → el modelo")
print("   ha encontrado estructura real, no ruido.")
print(f"
   Abandono en Inactivo vs VIP: mira los extremos de la tabla.")
print(f"   Si la diferencia es grande → clustering exitoso.")


---
## 📊 11 · Dashboard ejecutivo — Lo que le presentas al director


In [ ]:
# ── DASHBOARD EJECUTIVO ───────────────────────────────────────────────────────
seg_colores = {
    "💤 Perfil Inactivo" : GRAY,
    "⚙️  Perfil Medio"   : PURPLE,
    "🏆 Perfil VIP"      : GREEN,
}

fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])   # Gasto por segmento
ax2 = fig.add_subplot(gs[0, 1])   # Asistencias por segmento
ax3 = fig.add_subplot(gs[0, 2])   # Distribución N clientes
ax4 = fig.add_subplot(gs[1, 0])   # Tasa abandono
ax5 = fig.add_subplot(gs[1, 1:])  # Scatter Asistencias vs Gasto (las 2 más informativas)

# ── Gasto medio ───────────────────────────────────────────────────────────────
gasto = data.groupby("Segmento")["Gasto_Mensual_Extra"].mean().sort_values()
bar_c = [seg_colores.get(s, PURPLE) for s in gasto.index]
bars  = ax1.barh(gasto.index, gasto.values, color=bar_c, alpha=0.85, height=0.5)
for bar, val in zip(bars, gasto.values):
    ax1.text(val + 1, bar.get_y() + bar.get_height()/2,
             f"€{val:.0f}", va="center", fontsize=10, fontweight="bold", color=INK)
ax1.set_title("Gasto mensual extra", fontsize=11, fontweight="bold", color=INK)
ax1.set_xlabel("€ / mes", color=MUTED, fontsize=9)
ax1.tick_params(axis="y", labelsize=8)
ax1.invert_yaxis()

# ── Asistencias medias ────────────────────────────────────────────────────────
asist = data.groupby("Segmento")["Asistencias_Mes"].mean().sort_values()
bar_c2 = [seg_colores.get(s, PURPLE) for s in asist.index]
bars2  = ax2.barh(asist.index, asist.values, color=bar_c2, alpha=0.85, height=0.5)
for bar, val in zip(bars2, asist.values):
    ax2.text(val + 0.2, bar.get_y() + bar.get_height()/2,
             f"{val:.1f}", va="center", fontsize=10, fontweight="bold", color=INK)
ax2.set_title("Asistencias / mes", fontsize=11, fontweight="bold", color=INK)
ax2.set_xlabel("Visitas al mes", color=MUTED, fontsize=9)
ax2.tick_params(axis="y", labelsize=8)
ax2.invert_yaxis()

# ── Distribución clientes ─────────────────────────────────────────────────────
counts = data["Segmento"].value_counts()
bar_c3 = [seg_colores.get(s, PURPLE) for s in counts.index]
ax3.bar(range(len(counts)), counts.values, color=bar_c3, alpha=0.85, width=0.5)
ax3.set_xticks(range(len(counts)))
ax3.set_xticklabels([s.split(" ")[1] for s in counts.index], fontsize=9)
for i, val in enumerate(counts.values):
    ax3.text(i, val + 1, f"{val}
({val/len(data)*100:.0f}%)",
             ha="center", fontsize=9, color=INK, fontweight="bold")
ax3.set_title("Clientes por segmento", fontsize=11, fontweight="bold", color=INK)
ax3.set_ylabel("N clientes", color=MUTED, fontsize=9)
ax3.grid(axis="x", alpha=0)

# ── Tasa de abandono ─────────────────────────────────────────────────────────
abandono = data.groupby("Segmento")["Abandono"].mean() * 100
bar_c4   = [RED if v > 20 else (PURPLE if v > 8 else GREEN) for v in abandono.values]
bars4    = ax4.barh(abandono.index, abandono.values, color=bar_c4, alpha=0.85, height=0.5)
for bar, val in zip(bars4, abandono.values):
    ax4.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             f"{val:.1f}%", va="center", fontsize=10, fontweight="bold", color=INK)
ax4.set_title("Tasa de abandono real
(validación — no usada en clustering)",
              fontsize=11, fontweight="bold", color=INK)
ax4.set_xlabel("% abandono", color=MUTED, fontsize=9)
ax4.tick_params(axis="y", labelsize=8)
ax4.invert_yaxis()
ax4.text(0.03, 0.05, "⚠️ No usada en clustering
→ valida que los grupos son reales",
         transform=ax4.transAxes, fontsize=8, color=MUTED,
         bbox=dict(boxstyle="round,pad=0.3", facecolor=PURPLE_LITE, edgecolor="none"))

# ── Scatter Asistencias vs Gasto (las 2 más informativas del pairplot) ────────
for seg, color in seg_colores.items():
    mask = data["Segmento"] == seg
    ax5.scatter(
        data.loc[mask, "Asistencias_Mes"],
        data.loc[mask, "Gasto_Mensual_Extra"],
        color=color, alpha=0.65, s=55, label=seg, edgecolors="none"
    )
ax5.set_title("Asistencias vs Gasto mensual extra — Las 2 variables más discriminantes",
              fontsize=11, fontweight="bold", color=INK)
ax5.set_xlabel("Asistencias / mes", color=MUTED, fontsize=10)
ax5.set_ylabel("Gasto mensual extra (€)", color=MUTED, fontsize=10)
ax5.legend(fontsize=9, framealpha=0.95, edgecolor=GRAY)

fig.suptitle("Dashboard Ejecutivo — Segmentación de Clientes · Gimnasio · Clustering Jerárquico Ward",
             fontsize=14, fontweight="bold", color=INK, y=1.01)

plt.savefig("gym_clustering_dashboard.png", dpi=130, bbox_inches="tight",
            facecolor="white")
plt.show()


---
## 🎯 12 · Conclusión y plan de acción

### Lo que ha encontrado el algoritmo

El clustering jerárquico (Ward, 4 variables, k=3) ha identificado  
tres perfiles de cliente **sin conocer la etiqueta Abandono**:

| Segmento | Comportamiento | Acción recomendada |
|---|---|---|
| 💤 Perfil Inactivo | Pocas asistencias, gasto mínimo, alto riesgo churn | Campaña de reactivación urgente · Descuento en PT trial |
| ⚙️ Perfil Medio | Regularidad moderada, potencial sin explotar | Cross-sell: nutrición, clases premium, pack anual |
| 🏆 Perfil VIP | Alta frecuencia, alto gasto, muy fidelizado | Programa referidos · Acceso anticipado novedades |

### Por qué usar 4 variables es mejor que 2

Con 2 variables (V1) teníamos una visión parcial del cliente:  
solo antigüedad y gasto. Podías tener un cliente antiguo que ya no viene.

Con 4 variables (V2) el algoritmo distingue al cliente antiguo  
que sigue viniendo (VIP) del antiguo que ya no pisa el gimnasio (Inactivo).  
Esa diferencia **no era visible** con solo 2 dimensiones.

---

## 📋 Checklist final de buenas prácticas

```
✅ Seleccionar solo variables de comportamiento (no ID, no targets)
✅ Normalizar con StandardScaler antes del clustering
✅ Usar method="ward" como primera opción de linkage
✅ Calcular el linkage una sola vez en el notebook
✅ Ejes del dendrograma: observaciones / distancia de fusión
✅ Método del codo + aceleración para confirmar k
✅ fcluster para asignar etiquetas numéricas
✅ Heatmap de perfiles cuando hay más de 2 variables
✅ Pairplot para ver separación en todas las combinaciones
✅ Validar con la variable target (aunque no se usó en clustering)
✅ Terminar con plan de acción empresarial por segmento
```
